# Notebook 04: Multi-Head Attention

**Module:** 10 Transformers  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Derive multi-head attention
2. Implement MultiHeadAttention in PyTorch
3. Understand head splitting and concatenation
4. Build transformer encoder block

## Multi-Head Attention

Instead of one attention, run **h parallel heads** on different subspaces:

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W_O$$

$$\text{head}_i = \text{Attention}(QW_Q^i, KW_K^i, VW_V^i)$$

**Why multiple heads?** Different heads learn different relationships (local vs global, syntax vs semantics).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        Q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        out, w = scaled_dot_product_attention_torch(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_o(out), w

mha = MultiHeadAttention(64, 8)
x = torch.randn(2, 10, 64)
out, w = mha(x)
print(f'MHA output: {out.shape}, weights: {w.shape}')

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        attn_out, _ = self.mha(x)
        x = self.norm1(x + self.drop(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.drop(ff_out))
        return x

block = TransformerEncoderBlock(64, 8, 256)
print('Encoder block output:', block(torch.randn(2, 10, 64)).shape)

---

## Summary

Multi-head attention runs parallel heads; encoder block = MHA + FFN + residuals.

**Next:** [05_Positional_Encoding.ipynb](05_Positional_Encoding.ipynb)